In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchaudio
from tqdm.notebook import trange, tqdm
from datasets import load_dataset, Audio
import IPython.display as ipd

In [2]:
# --- Configuration ---
SAMPLE_RATE = 8000
FRAME_SIZE = 512
LATENT_DIM = 32
NUM_EMBEDDINGS = 512 # Dictionary of sound features
COMMITMENT_COST = 0.25
decay = 0.99

In [3]:
class ResidualBlock(nn.Module): # ResNet (CMSIS eq.: arm_add_f32)
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size=1)
        )

    def forward(self, x):
        return x + self.block(x)

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super(VectorQuantizer, self).__init__()
        self._embedding_dim = embedding_dim
        self._num_embeddings = num_embeddings

        # The Codebook: A dictionary of 512 distinct "sound vectors"
        self._embedding = nn.Embedding(self._num_embeddings, self._embedding_dim)
        self._embedding.weight.data.uniform_(-1/self._num_embeddings, 1/self._num_embeddings)
        self._commitment_cost = commitment_cost

    def forward(self, inputs):
        # inputs: [Batch, Channels (Latent), Time] -> Permute to [Batch, Time, Latent]
        inputs = inputs.permute(0, 2, 1).contiguous()

        # Flatten input
        input_shape = inputs.shape
        flat_input = inputs.view(-1, self._embedding_dim)

        # Calculate distances between input and codebook
        distances = (torch.sum(flat_input**2, dim=1, keepdim=True)
                     + torch.sum(self._embedding.weight**2, dim=1)
                     - 2 * torch.matmul(flat_input, self._embedding.weight.t()))

        # Encoding: Find the nearest codebook index per input
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)

        # Quantize: Replace input with the nearest codebook vector
        quantized = self._embedding(encoding_indices).view(input_shape)

        # Loss computation
        e_latent_loss = torch.mean((quantized.detach() - inputs)**2)
        q_latent_loss = torch.mean((quantized - inputs.detach())**2)
        loss = q_latent_loss + self._commitment_cost * e_latent_loss

        quantized = inputs + (quantized - inputs).detach()

        # Permute back to [Batch, Channels, Time]
        return loss, quantized.permute(0, 2, 1).contiguous(), encoding_indices

class AudioVQVAE(nn.Module):
    def __init__(self):
        super(AudioVQVAE, self).__init__()

        # Encoder
        # Input: 1 x 512
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=4, stride=2, padding=1), # -> 32 x 256
            nn.ReLU(),
            ResidualBlock(32),
            nn.Conv1d(32, 64, kernel_size=4, stride=2, padding=1), # -> 64 x 128
            nn.ReLU(),
            ResidualBlock(64),
            nn.Conv1d(64, LATENT_DIM, kernel_size=4, stride=2, padding=1), # -> 32 x 64
        )

        # The Vector Quantizer Layer
        self.vq = VectorQuantizer(NUM_EMBEDDINGS, LATENT_DIM, COMMITMENT_COST)

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(LATENT_DIM, 64, kernel_size=4, stride=2, padding=1),
            ResidualBlock(64),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),
            ResidualBlock(32),
            nn.ReLU(),
            nn.ConvTranspose1d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh() # Output -1 to 1 (Mu-Law range)
        )

    def forward(self, x):
        z = self.encoder(x)
        loss, quantized, indices = self.vq(z)
        x_recon = self.decoder(quantized)
        return loss, x_recon, indices

    # Helper to decode from indices (Simulates Receiver Side)
    def decode_from_indices(self, indices):
        # indices shape: [Batch, Time_Steps]
        # Map indices to vectors
        quantized = self.vq._embedding(indices)
        # Reshape for decoder: [Batch, Time, Latent] -> [Batch, Latent, Time]
        quantized = quantized.permute(0, 2, 1)
        return self.decoder(quantized)

---
## Data

In [4]:
# --- Configuration ---
OUTPUT_FILE = "data/processed_training_data_3000.npy"
NUM_SUBSET = 3000

In [5]:
def mu_law_encoding(x, mu=255): # Compresses
    return torch.sign(x) * torch.log1p(mu * torch.abs(x)) / torch.log1p(torch.tensor(mu))

In [6]:
dataset = load_dataset('nguyenvulebinh/libris_clean_100', split='train.clean.100')
dataset = dataset.shuffle(seed=42).select(range(NUM_SUBSET)) # Selecting subset
dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

processed_frames = []
for item in tqdm(dataset):
    audio_array = item['audio']['array']

    # To Tensor
    waveform = torch.from_numpy(audio_array).float().unsqueeze(0) # [1, Time]

    # Normalize (-1 to 1)
    max_val = torch.max(torch.abs(waveform))
    if max_val > 0:
        waveform = waveform / max_val

    waveform = mu_law_encoding(waveform)

    # Slicing
    num_frames = waveform.shape[1] // FRAME_SIZE
    if num_frames == 0:
        continue

    # Truncate [N_frames, 1, FRAME_SIZE]
    waveform = waveform[:, :num_frames * FRAME_SIZE]
    frames = waveform.view(num_frames, 1, FRAME_SIZE)

    processed_frames.append(frames.numpy())

# Concatenate all frames
all_data = np.concatenate(processed_frames, axis=0)
np.save(OUTPUT_FILE, all_data)

  0%|          | 0/3000 [00:00<?, ?it/s]

---
## Training

In [6]:
# --- Configuration ---
BATCH_SIZE = 1024
NUM_WORKERS = 4
LR = 1e-3

In [7]:
dataset = torch.Tensor(np.load(OUTPUT_FILE)); print(f'Loaded dataset: {dataset.shape}')
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu'); print(f'Using device: {device}')
model = AudioVQVAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

print('Starting Training...')
for epoch in trange(15):
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        vq_loss, recon, _ = model(batch)

        # Reconstruction Loss
        recon_loss = F.mse_loss(recon, batch)

        loss = recon_loss + vq_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch}: Loss {total_loss/len(loader)}')

Loaded dataset: torch.Size([590587, 1, 512])
Using device: mps
Starting Training...


  0%|          | 0/15 [00:00<?, ?it/s]

/Users/loicduchesne/Library/CloudStorage/OneDrive-Personal/Projects/AudioVAE/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 0: Loss 0.84149097600884
Epoch 1: Loss 0.10323102537226223
Epoch 2: Loss 0.08607139094915092
Epoch 3: Loss 0.07816512043389331
Epoch 4: Loss 0.06901541168267226
Epoch 5: Loss 0.0638334026343901
Epoch 6: Loss 0.05902724886761794
Epoch 7: Loss 0.055721586031041895
Epoch 8: Loss 0.05420234557273483
Epoch 9: Loss 0.053450067028842314
Epoch 10: Loss 0.05293715613501944
Epoch 11: Loss 0.052508844705088716
Epoch 12: Loss 0.052241858036823866
Epoch 13: Loss 0.05194704442493217
Epoch 14: Loss 0.051587186831225554


In [8]:
# Save Weights
torch.save(model.state_dict(), 'outputs/vqvae_weights.pth')

---
## Testing

In [12]:
import soundfile as sf

In [13]:
# --- Configuration ---
SECONDS = 10

In [14]:
def mu_law_expansion(x, mu=255):
    x = np.array(x)
    return np.sign(x) * (1 / mu) * ((1 + mu) ** np.abs(x) - 1)

In [15]:
# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
model = AudioVQVAE().to(device)

state_dict = torch.load('outputs/vqvae_weights.pth', map_location=device, weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [16]:
# Testing
model.eval()

# 1. Load a fresh clip
ds = load_dataset('nguyenvulebinh/libris_clean_100', split='train.clean.100', streaming=True)
ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
dataset_item = next(iter(ds))

# Get Audio
audio_array = dataset_item['audio']['array']

# Trim
max_len = SAMPLE_RATE * SECONDS
if len(audio_array) > max_len:
    audio_array = audio_array[:max_len]

# 2. Preprocessing
waveform = torch.from_numpy(audio_array).float()
waveform = waveform / torch.max(torch.abs(waveform)) # Normalize

mu = 255
waveform_mu = mu_law_encoding(waveform, mu)

# 3. Chunking
num_frames = len(waveform_mu) // FRAME_SIZE

# Create batch of [N, 1, 512]
input_tensor = torch.zeros(num_frames, 1, FRAME_SIZE)
for i in range(num_frames):
    input_tensor[i, 0, :] = waveform_mu[i*FRAME_SIZE : (i+1)*FRAME_SIZE]

input_tensor = input_tensor.to(device)

# 4. Inference
with torch.no_grad():
    _, reconstructed_batch, _ = model(input_tensor)

# 5. Post-processing
reconstructed_audio = reconstructed_batch.cpu().numpy().flatten()
original_audio_mu = input_tensor.cpu().numpy().flatten()

# Expand back to linear audio
recon_linear = mu_law_expansion(reconstructed_audio)
orig_linear = mu_law_expansion(original_audio_mu)

# 6. Playback
print(f"--- Original (Mu-Law Encoded 8kHz) ---")
ipd.display(ipd.Audio(orig_linear, rate=SAMPLE_RATE))

print(f"--- VQ-VAE Reconstructed ---")
ipd.display(ipd.Audio(recon_linear, rate=SAMPLE_RATE))

--- Original (Mu-Law Encoded 8kHz) ---


--- VQ-VAE Reconstructed ---


In [17]:
# Saving samples
sf.write('outputs/examples/original.flac', orig_linear, SAMPLE_RATE)
sf.write('outputs/examples/reconstructed.flac', recon_linear, SAMPLE_RATE)